In [114]:
from pathlib import Path
import sys
repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
repo_root

WindowsPath('C:/Users/leungk/OneDrive - EllisDon Corporation/Documents/Other/github_repos/CBG_analysis')

# Validation and Guardrails
Lightweight assertions to catch common failure modes in parsing, defaults, units, and outputs so glucose-focused analyses stay trustworthy after pipeline runs.

In [115]:
from pathlib import Path
import pandas as pd
from IPython.display import display
import importlib
from src import units, defaults, io_excel, run_pipeline, validate, aggregate

repo_root = Path('..').resolve()
data_dir = repo_root / 'data' / 'outputs'
excel_path = repo_root / 'data' / 'source_data' / '20251218_Trudy_Meals.xlsx'
defaults_path = repo_root / 'config' / 'defaults_food_items.yaml'
grams_overrides_path = repo_root / 'config' / 'grams_overrides.yaml'
api_keys_path = repo_root / 'config' / 'api_keys.json'

# Ensure latest code is loaded when running in-place
importlib.reload(units)
importlib.reload(defaults)
importlib.reload(aggregate)
importlib.reload(validate)

print("[Setup] Using repo_root", repo_root)
print("[Setup] Outputs directory", data_dir)

[Setup] Using repo_root C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis
[Setup] Outputs directory C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\data\outputs


In [116]:
# Unit-level tests for parsing and defaults
print('--- Unit Tests: Fractions and Parsing ---')
print('Inputs: simple fractions, mixed numbers, and a quantified food string. Expected: numeric quantities + canonical names/units.')
assert units.parse_fraction('1/2') == 0.5
print('parse_fraction half: OK (0.5 from "1/2")')
assert units.parse_fraction('1 1/2') == 1.5
print('parse_fraction mixed: OK (1.5 from "1 1/2")')
p = units.parse_quantity_unit('1.5 oz salmon')
assert p.qty_numeric == 1.5
assert (p.unit_std == 'oz') or (p.unit_raw and 'oz' in p.unit_raw.lower())
assert 'salmon' in p.food_name_std
print('parse_quantity_unit: OK (qty=1.5, unit~oz, food contains salmon)')
g, reason = units.convert_to_grams(1, 'cup', 'blueberries', grams_per_cup_map=units.GRAMS_PER_CUP)
assert round(g, 1) == round(units.GRAMS_PER_CUP['blueberries'], 1)
print('convert_to_grams blueberries: OK (uses grams_per_cup mapping, no 100g fallback)')
print('\n--- Unit Tests: Defaults ---')
cfg = defaults.load_defaults_config(defaults_path)
df = pd.DataFrame([{'food_name_std': 'blueberries', 'qty_numeric': None, 'unit_std': None},
                   {'food_name_std': 'flaxseed', 'qty_numeric': None, 'unit_std': None}])
df = defaults.apply_default_rules(df, cfg)
assert df.loc[0, 'grams_override'] == 100 or df.loc[0, 'qty_numeric'] == 100
print('defaults blueberries: OK (falls back to 100g when qty/unit missing)')
assert df.loc[1, 'qty_numeric'] == 3 and df.loc[1, 'unit_std'] == 'tsp'
print('defaults flaxseed: OK (applies 3 tsp rule)')
print('\nOutputs: quantities/units normalized; defaults applied where missing. All unit-level tests passed.')

--- Unit Tests: Fractions and Parsing ---
Inputs: simple fractions, mixed numbers, and a quantified food string. Expected: numeric quantities + canonical names/units.
parse_fraction half: OK (0.5 from "1/2")
parse_fraction mixed: OK (1.5 from "1 1/2")
parse_quantity_unit: OK (qty=1.5, unit~oz, food contains salmon)
convert_to_grams blueberries: OK (uses grams_per_cup mapping, no 100g fallback)

--- Unit Tests: Defaults ---
defaults blueberries: OK (falls back to 100g when qty/unit missing)
defaults flaxseed: OK (applies 3 tsp rule)

Outputs: quantities/units normalized; defaults applied where missing. All unit-level tests passed.


In [117]:
# Integration checks using latest pipeline outputs
print('--- Integration Tests: Pipeline outputs and schema sanity ---')
print('Inputs: latest CSV outputs (items, meal_features, model_table). If missing, pipeline reruns.')
expected_files = [
    data_dir / 'food_items.csv',
    data_dir / 'meal_features.csv',
    data_dir / 'model_table.csv',
]
missing = [p for p in expected_files if not p.exists()]
if missing:
    print('Missing outputs detected -> running pipeline to regenerate CSVs...')
    summary = run_pipeline.run_pipeline(excel_path, api_keys_path, defaults_path, data_dir)
    print('Pipeline summary after regeneration:', summary)
else:
    print('All expected output CSVs found; using existing files.')
items = pd.read_csv(data_dir / 'food_items.csv', parse_dates=['datetime'])
meal_features = pd.read_csv(data_dir / 'meal_features.csv', parse_dates=['datetime'])
model_table = pd.read_csv(data_dir / 'model_table.csv', parse_dates=['datetime'])

print(f'items rows: {len(items)}')
print(f'meal_features rows: {len(meal_features)}')
print(f'model_table rows: {len(model_table)}')

assert len(items) > 0
print('items presence: OK')
assert (items['grams_final'] > 0).all()
print('grams_final positive: OK (no zero/negative grams)')
assert items['meal_id'].notna().all()
print('meal_id present on items: OK')
assert meal_features['datetime'].notna().all(), 'meal_features has NaT datetimes'
print('meal_features datetime present: OK')
assert meal_features['meal_id'].is_unique
print('meal_id uniqueness: OK (no duplicates)')
assert 'cbg_prev_same_day' in model_table.columns
print('cbg_prev_same_day present: OK (lag feature available)')

assumed_rate = items.get('assumed_100g_flag', False).mean()
print(f'assumed_rate: {assumed_rate:.3f} (lower is better; reflects 100g fallbacks)')
assert assumed_rate < 1.0, 'all items assumed; check parsing/defaults'
print('\nOutputs: integration checks passed; tables ready for downstream analysis.')

--- Integration Tests: Pipeline outputs and schema sanity ---
Inputs: latest CSV outputs (items, meal_features, model_table). If missing, pipeline reruns.
All expected output CSVs found; using existing files.
items rows: 759
meal_features rows: 205
model_table rows: 205
items presence: OK
grams_final positive: OK (no zero/negative grams)
meal_id present on items: OK
meal_features datetime present: OK
meal_id uniqueness: OK (no duplicates)
cbg_prev_same_day present: OK (lag feature available)
assumed_rate: 0.038 (lower is better; reflects 100g fallbacks)

Outputs: integration checks passed; tables ready for downstream analysis.


In [118]:
# Parsing harness regression cases
print('--- Parsing harness: expected quantity/unit/food_name/grams outcomes ---')
print('Inputs: canonical problem phrases (fractions, mixed numbers, misspellings). Process: validate.validate_parsing_examples with defaults + grams overrides. Output: table with parsed fields and 100g fallback flag.')
cases_df = validate.validate_parsing_examples(defaults_path=defaults_path, grams_overrides_path=grams_overrides_path)
display(cases_df)

--- Parsing harness: expected quantity/unit/food_name/grams outcomes ---
Inputs: canonical problem phrases (fractions, mixed numbers, misspellings). Process: validate.validate_parsing_examples with defaults + grams overrides. Output: table with parsed fields and 100g fallback flag.


,food_text_raw,qty_raw,qty_numeric,unit_std,food_name_std,grams_final,used_100g_fallback
0,2 eggs,2,2.00,count,eggs,100.00000,False
1,1 oz cheese,1,1.00,oz,cheese,28.34950,False
2,1 tsp flax seed,1,1.00,tsp,flax seed seed,4.92892,False
3,2 slices of pizza,2,2.00,count,pizza,250.00000,False
4,2 cups veggies,2,2.00,cup,vegetables,300.00000,False
5,1/4 cup mixed nuts,1/4,0.25,cup,mixed nuts,37.50000,False
6,1 4 cup mixed nuts,1/4,0.25,cup,mixed nuts,37.50000,False
7,1/2 slice of bread,1/2,0.50,count,bread,14.00000,False
8,1 2 slice of bread,1/2,0.50,count,bread,14.00000,False
9,brocoli,None,100.00,g,broccoli,100.00000,False


In [119]:
# Run lightweight parsing tests (mirrors tests/test_parsing.py)
print('--- Scripted parsing tests: mirrors tests/test_parsing.py ---')
print('Inputs: same cases as unit script; Process: executes test file directly; Output: stdout pass/fail traces.')
import runpy
print('Running tests/test_parsing.py ...')
runpy.run_path(str(repo_root / 'tests' / 'test_parsing.py'), run_name='__main__')
print('tests/test_parsing.py completed')

--- Scripted parsing tests: mirrors tests/test_parsing.py ---
Inputs: same cases as unit script; Process: executes test file directly; Output: stdout pass/fail traces.
Running tests/test_parsing.py ...
[RUN] Starting parsing tests (script mode)
[BG] Parsing BG values with single and dual readings
[TEST] BG dual reading avg: 6.4 and 5.8 -> 6.1 ... OK
[TEST] BG single reading: 4.9 -> 4.9 ... OK
[TEST] BG empty string -> NaN ... OK
[UNITS] Parsing compact number+word items (count units)
[TEST] 2eggs -> qty=2, unit missing, food=eggs ... OK
[COMPOUND] Evenly split qty across two foods
[TEST] 1 cup beef brisket/tendon -> two items at 0.5 cup each ... OK
[COMPOUND] Special skyr/berries/seeds/kiwi expansion with defaults
[TEST] 1/4 cup skyr/blueberries/flax/hemp/kiwi expanded to 5 items with defaults ... OK
[RUN] All parsing tests passed.
tests/test_parsing.py completed


In [123]:
# Full test suite via pytest (runs from repo root to find tests/fixtures)
import subprocess, sys

cmd = [sys.executable, "-m", "pytest", "-q", "tests"]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, text=True, cwd=repo_root, capture_output=True)
print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)
if result.returncode != 0:
    raise SystemExit("Pytest failed; check notebook cell output above.")

Running: c:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\.venv\Scripts\python.exe -m pytest -q tests
......                                                                   [100%]
6 passed in 1.48s


Return code: 0
